# Anima — Colab A100 cache FACTORY (thin shell over the repo runner)

Builds the expensive VAE-latent + Qwen-3 text-embed cache for the **full** dataset on the A100's big,
fast, **ephemeral** local-scratch SSD, and pushes it to HF Hub so a training box (e.g. the RTX PRO 6000)
pulls + reconstructs it. It does **not** train.

**Why this notebook barely has any code.** Colab can't hot-swap a notebook — every session is a fresh
notebook + runtime, and changing logic means re-pasting cells by hand. So all the behaviour lives in the
**repo** (`geolip_anima_trainer.cache_factory.CacheFactory`), and this notebook is just `clone → install →
a handful of `f.<step>()` calls`. **To change anything, edit the repo and `git pull` — the notebook stays
as-is.** State is reconstructed from HF + a deterministic config on every fresh runtime, so a reset just
means re-running the same cells.

**Before you start:** add your HF **WRITE** token to Colab Secrets (🔑) as `HF_TOKEN` — the factory must
push (the scratch is wiped on disconnect; the HF push is the durable copy).

> **License — NON-COMMERCIAL.** Anima + any derivative cache/LoRA inherit CircleStone NC + the NVIDIA Open
> Model License (Cosmos derivative).

## 0 · Confirm the GPU + disks

Expect an **A100** (sm_80, ~80 GB) and a big **local-scratch** line in the resources panel. `df -h` shows
the scratch mount the factory auto-detects (the ~368 GB filesystem that isn't `/` or Drive).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h | grep -vE 'tmpfs|overlay|udev'

## 1 · Bootstrap — clone + install (→ restart ONCE)

Clones this repo + diffusion-pipe and installs deps. The install logic lives in the repo
(`anima_colab.install`) so a `git pull` changes it with no notebook edit. **Run this cell; when the kernel
restarts, run it again** (the second time it detects the install and skips, so no restart), then continue.

In [ ]:
import os, sys, subprocess
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO):           # ensure_repo (git pull + diffusion-pipe) + pip; True the 1st time
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · (after restart) Run the cache factory

Each cell is one `CacheFactory` step — all the logic is in the repo. `setup()` auto-detects the scratch SSD,
puts `HF_HOME`/`TMPDIR`/`DATA_ROOT` on it (symlinked to a portable `/workspace/anima_data` so the cache is
reusable on the training box), authenticates HF, checks the GPU, and downloads the model.

`limit=None` caches the **full** set; set e.g. `limit=40000` for a subset. Re-running any cell is safe
(idempotent); a fresh runtime just re-runs them and resumes from HF.

In [ ]:
from geolip_anima_trainer.cache_factory import CacheFactory
f = CacheFactory(limit=None)        # full set; edit limit= for a subset, cache_repo= to rename the HF repo
f.setup()                           # scratch + HF_HOME, HF auth, GPU check, model download

In [ ]:
f.prepare_dataset()                 # resume from HF, or extract + store the index; prunes the source parquet

In [ ]:
f.build_configs()                   # two dataset tomls + a lora toml per phase

In [ ]:
f.run_cache()                       # cache both phases; periodic + final push to HF (the durable copy)

In [ ]:
f.status()                          # disk usage + cached latent/text counts (run anytime, even mid-cache)

## Notes

- **One-shot:** `CacheFactory(limit=None).run_all()` does setup → prepare → build → cache in one call.
- **Scratch is ephemeral** (wiped on disconnect) — the periodic HF push is the durability, not the disk. On
  **Colab** the real protection is that push (a drop loses ≤ the in-flight 10 GB shard, and a re-run resumes
  via `trust_cache`); the in-kernel keepalive is the *wrong* lever here (Colab idle = browser-UI, not GPU).
- **Reuse on the training box (RTX PRO 6000):** the cache is keyed by absolute image paths, and `setup()`
  pins them to a portable `/workspace/anima_data` (symlinked to scratch). On the training box, make
  `/workspace/anima_data` a real dir and `cache_pull` reconstructs there — same paths → the cache resumes.
- **Updating logic = `git pull`.** Step 1 pulls the latest repo every run; nothing here needs editing.
- **Colab Pro+ limits:** ~24 h hard cap (gated on compute units); background execution is best-effort —
  lean on the HF push, not on the session surviving. ~8 h for the full set burns ~50 units.